# 富途 Stage1 模拟盘恢复：实际成交收益审计

核对 2026-08-03 $100,000 新周期到当前的真实已成交流水和持仓收益。使用专属 qlib provider、Data API、原生风险分析与 Recorder；不运行策略、不生成订单，不重置任何运行状态。现金调整未取得日期明细时，不虚构净收益日度轨迹。

In [1]:
from pathlib import Path
import json, sys, os
# This repository uses the supported local Qlib/MLflow file recorder.
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")
ROOT = Path.cwd()
while not (ROOT / 'SPEC.md').exists():
    if ROOT == ROOT.parent:
        raise RuntimeError('Repository root not found')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
CARD = ROOT / 'production/stage1_recovery_20260928'
PROVIDER = ROOT / 'data/qlib/stage1_recovery_20260928'
# Prefer this task's durable final observation, retaining the original 16:15 anchor.
prior = json.loads((CARD / 'summary.json').read_text())
archived_source = Path(prior['recorder_local_dir']) / 'artifacts/source'
EVIDENCE = ROOT / 'runtime/soxl/stage1_futu_sim/recovery_evidence/20260928'
SOURCE = EVIDENCE / 'final_observation'
if not SOURCE.exists():
    SOURCE = archived_source / 'broker'
HISTORY = archived_source / 'historical'
ANCHOR = EVIDENCE if EVIDENCE.exists() else archived_source / 'anchor'
CONFIRMATION = EVIDENCE / 'net_cash_adjustment_confirmation.json'
if not CONFIRMATION.exists():
    CONFIRMATION = archived_source / 'confirmation/net_cash_adjustment_confirmation.json'
OBSERVATION = json.loads((SOURCE / 'observation_manifest.json').read_text())
BALANCE_QUERY = next(item for item in OBSERVATION['calls'] if item['query'] == 'STOCK_AND_OPTION_balance')
AS_OF = '2026-09-25'
from src.analysis.stage1_recovery import build_recovery_provider, run_recovery_audit
provider_manifest = build_recovery_provider(SOURCE, PROVIDER, as_of=AS_OF)
assert provider_manifest['status'] == 'ok'
provider_manifest

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[61132:MainThread](2026-09-28 17:03:15,295) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


2026/09/28 17:03:15 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


[61132:MainThread](2026-09-28 17:03:15,762) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[61132:MainThread](2026-09-28 17:03:15,762) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/stage1_recovery_20260928')}


...

[61132:MainThread](2026-09-28 17:03:15,769) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/stage1_recovery_20260928')}


[61132:MainThread](2026-09-28 17:03:15,770) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[61132:MainThread](2026-09-28 17:03:15,770) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[61132:MainThread](2026-09-28 17:03:15,771) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/stage1_recovery_20260928')}


{'provider_uri': '/Users/rziiiii/dev/workspace/data/qlib/stage1_recovery_20260928',
 'market': 'all',
 'region': 'us',
 'freq': 'day',
 'as_of': '2026-09-25',
 'source': 'Futu request_history_kline K_DAY AuType.NONE; observed SIM audit',
 'instruments': ['SOXL', 'SOXX'],
 'fields': ['$open', '$close', '$high', '$low', '$volume', '$factor'],
 'calendar_start': '2026-08-03',
 'calendar_end': '2026-09-25',
 'max_datetime': '2026-09-25',
 'row_count': 78,
 'feature_file_count': 12,
 'audit_snapshot_hash': '379b64e168a3a429',
 'operation': 'qlib_store_build',
 'status': 'ok',
 'hash': 'c6149d758306aa05',
 'source_api': ['qlib.init',
  'qlib.data.storage.file_storage.FileFeatureStorage',
  'qlib.data.D.calendar',
  'qlib.data.D.instruments',
  'qlib.data.D.features']}

In [2]:
import qlib
from qlib.data import D
from qlib.workflow import R
qlib.init(provider_uri=str(PROVIDER), region='us', kernels=1)
calendar = D.calendar(start_time='2026-08-03', end_time=AS_OF, freq='day')
features = D.features(['SOXL', 'SOXX'], ['$close'], start_time='2026-08-03', end_time=AS_OF)
assert len(calendar) == 39 and not features.isna().any().any()
summary = run_recovery_audit(source_dir=SOURCE, historical_dir=HISTORY,
    provider_uri=PROVIDER, card_dir=CARD, as_of=AS_OF,
    account_observed_at=BALANCE_QUERY['completed_at'], cash_confirmation_path=CONFIRMATION,
    anchor_source_dir=ANCHOR)
summary['reconciliation'], summary['metrics']

[61132:MainThread](2026-09-28 17:03:15,781) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[61132:MainThread](2026-09-28 17:03:15,782) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[61132:MainThread](2026-09-28 17:03:15,782) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/stage1_recovery_20260928')}


[61132:MainThread](2026-09-28 17:03:15,808) INFO - qlib.workflow - [exp.py:258] - Experiment 804525797666560748 starts running ...


[61132:MainThread](2026-09-28 17:03:15,879) INFO - qlib.workflow - [recorder.py:345] - Recorder 187a75db4b88407db41579e4f0b60bf1 starts running under Experiment 804525797666560748 ...


[61132:MainThread](2026-09-28 17:03:16,611) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | waiting `async_log` Done


({'initial_equity': 100000.0,
  'inception_date': '2026-08-03',
  'fill_count': 20,
  'historical_fill_count_matched': 15,
  'last_fill_time_new_york': '2026-08-12 09:31:51',
  'positions': {'SOXL': 200.0, 'SOXX': 98.0},
  'buy_value': 86662.11000000002,
  'sell_value': 19591.355,
  'trade_only_cash': 32929.245,
  'broker_cash': 32959.769,
  'unclassified_net_cash_adjustment': 30.524,
  'cash_adjustment_interpretation': 'Aggregate residual; timing/components unavailable. Not actual per-fill fees.',
  'positions_reconciled': True,
  'account_equity_reconciled': True,
  'external_cash_flows': 'excluded_by_scoped_user_confirmation',
  'cash_adjustment_classification': 'unallocated_net_investment_cash_flow',
  'observation_lineage': 'Same cash, share quantities and all filled-order fields as confirmed anchor; later equity changes are market marks only.',
  'corporate_action_evidence': [{'symbol': 'SOXL',
    'ex_div_date': '2026-09-22',
    'prior_shares': 200.0,
    'per_cash_div': 0.0929

In [3]:
# Read primary evidence back from the qlib Recorder, not from loose report CSVs.
with R.uri_context(uri=str((CARD / 'qlib_recorder').resolve())):
    recorder = R.get_recorder(recorder_id=summary['recorder_id'],
        experiment_name=summary['experiment_name'])
    report = recorder.load_object('broker_audit/observed_holdings_valuation.pkl')
    risk = recorder.load_object('broker_audit/risk_analysis.pkl')
display(report.tail(), risk)
from IPython.display import Markdown, display
display(Markdown((CARD / 'conclusion.md').read_text()))

,trade_only_cash,SOXL_shares,SOXX_shares,SOXL_close,SOXX_close,equity_before_unclassified_cash_adjustments,fill_count,return
date,,,,,,,,
2026-09-21,32929.245,200.0,98.0,141.93,559.34,116130.565,0,0.056654
2026-09-22,32929.245,200.0,98.0,151.95,572.78,119451.685,0,0.028598
2026-09-23,32929.245,200.0,98.0,146.25,565.72,117619.805,0,-0.015336
2026-09-24,32929.245,200.0,98.0,146.33,566.07,117670.105,0,0.000428
2026-09-25,32929.245,200.0,98.0,151.45,572.68,119341.885,0,0.014207


,risk
mean,0.004544
std,0.030041
annualized_return,2.134731
information_ratio,2.401267
max_drawdown,-0.136649


# 富途 Stage1 模拟盘实际收益审计

从 2026-08-03 的 $100,000 起点核对 20 笔已成交订单，历史15笔全部匹配，当前持仓 SOXL 200 股、SOXX 98 股。最后成交为纽约时间 2026-08-12 09:31:51；此后收益来自未再调仓的持仓，不代表模型持续运行。

按 2026-09-25 正式收盘 SOXL $151.45、SOXX $572.68，使用查询时现金重估权益 $119,372.409，收益 19.372409%。2026-09-28T17:00:13.647+08:00 至 2026-09-28T17:00:15.610+08:00 查询窗口账户权益 $115,802.529，收益 15.802529%；两者是不同价格时点。行情快照 update_time 不能当作 last_price 的盘前成交时间。

买入 $86,662.110、卖出 $19,591.355，成交现金流推算现金 $32,929.245，实际现金 $32,959.769，差额 +$30.524 为未分类净现金调整；未取得明细，不能称为逐笔手续费或直接认定分红。企业行动资料显示本周期理论税前现金分配合计 $50.4349（SOXL 9月22日 $0.0929×200，SOXX 9月15日 $0.32505×98）；减去净现金调整后差额 $19.9109，与费用等扣款相容，但不是已核实实际手续费。SIM 费用查询及资金流水接口不支持，支付日期、实入金额与扣款无法逐笔确认。因此收盘重估不是已核实的9月25日历史结单。未分配该调整的成交现金流口径收盘权益 $119,341.885，收益 19.341885%。

qlib Data API 读取 39 个交易日；日度曲线排除时间未知现金调整，qlib 几何风险分析最大回撤 -13.6649%。风险数只适用于此重建口径。基准是同起始成交价的假设买入持有价格收益，未计费用及分配款。

本卡只审计实际模拟账户，不训练、不晋升、不补造遗漏交易、不修改运行账本。下一步补齐现金流水，再与修复后模型反事实回放分开比较。

Recorder: `187a75db4b88407db41579e4f0b60bf1`。明细、原始快照、哈希及风险分析保存在 qlib recorder；专属 provider 来源为不复权日K。

补充确认（记录于 2026-09-28T16:40:34+08:00）：用户原文“没有加减资金，也没有再次重置”。作用范围为8月3日初始10万美元至本次9月28日账户快照，因此可以排除该区间外部加减资金及重置，原始确认后已核对现金、持仓及全部成交不变，查询时权益相对初始本金的15.802529%可明确为本周期账户总收益。现金差额 +$30.524 归为未分配的净投资现金流；股息、费用与其他投资现金项目仍未逐笔分解，总手续费未知，历史到账日期不分配。最近收盘重估19.372409%的价格/现金时点限制不变。


# 富途 Stage1 模拟盘实际收益审计

从 2026-08-03 的 $100,000 起点核对 20 笔已成交订单，历史15笔全部匹配，当前持仓 SOXL 200 股、SOXX 98 股。最后成交为纽约时间 2026-08-12 09:31:51；此后收益来自未再调仓的持仓，不代表模型持续运行。

按 2026-09-25 正式收盘 SOXL $151.45、SOXX $572.68，使用查询时现金重估权益 $119,372.409，收益 19.372409%。2026-09-28T17:00:13.647+08:00 至 2026-09-28T17:00:15.610+08:00 查询窗口账户权益 $115,802.529，收益 15.802529%；两者是不同价格时点。行情快照 update_time 不能当作 last_price 的盘前成交时间。

买入 $86,662.110、卖出 $19,591.355，成交现金流推算现金 $32,929.245，实际现金 $32,959.769，差额 +$30.524 为未分类净现金调整；未取得明细，不能称为逐笔手续费或直接认定分红。企业行动资料显示本周期理论税前现金分配合计 $50.4349（SOXL 9月22日 $0.0929×200，SOXX 9月15日 $0.32505×98）；减去净现金调整后差额 $19.9109，与费用等扣款相容，但不是已核实实际手续费。SIM 费用查询及资金流水接口不支持，支付日期、实入金额与扣款无法逐笔确认。因此收盘重估不是已核实的9月25日历史结单。未分配该调整的成交现金流口径收盘权益 $119,341.885，收益 19.341885%。

qlib Data API 读取 39 个交易日；日度曲线排除时间未知现金调整，qlib 几何风险分析最大回撤 -13.6649%。风险数只适用于此重建口径。基准是同起始成交价的假设买入持有价格收益，未计费用及分配款。

本卡只审计实际模拟账户，不训练、不晋升、不补造遗漏交易、不修改运行账本。下一步补齐现金流水，再与修复后模型反事实回放分开比较。

Recorder: `187a75db4b88407db41579e4f0b60bf1`。明细、原始快照、哈希及风险分析保存在 qlib recorder；专属 provider 来源为不复权日K。

补充确认（记录于 2026-09-28T16:40:34+08:00）：用户原文“没有加减资金，也没有再次重置”。作用范围为8月3日初始10万美元至原始 2026-09-28T16:15:00+08:00 账户快照，因此可以排除该区间外部加减资金及重置，原始确认后已核对现金、持仓及全部成交不变，查询时权益相对初始本金的15.802529%可明确为本周期账户总收益。现金差额 +$30.524 归为未分配的净投资现金流；股息、费用与其他投资现金项目仍未逐笔分解，总手续费未知，历史到账日期不分配。最近收盘重估19.372409%的价格/现金时点限制不变。
